# AOD Regional Comparison — North / Central / South Vietnam

Extends the global MODIS vs Himawari comparison by breaking stations into **three geographic regions**  
derived from a spatial join against GADM Level-1 (province) boundaries.

**Two main questions**
1. **Correlation by region** — Does MODIS vs Himawari agreement differ across North, Central, and South Vietnam?
2. **Coincident data gaps** — Do the periods where MODIS and Himawari have the most missing data align with each other, or are the gaps sensor-specific?

## 0 · Imports & Configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, RANSACRegressor
import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings('ignore')
# 1. ACADEMIC STYLE SETUP
sns.set_theme(style='ticks', font_scale=1.1)

plt.rcParams.update({
    'figure.dpi': 300,
    'figure.facecolor': 'white',
    'font.family': 'serif',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 14
})

# ── Data paths ─────────────────────────────────────────────────────────────
MODIS_DIR  = Path('/home/slow_data/Air_Quality/MODIS_MCD19A2/stations_aod_v3')
HIMA_DIR   = Path('/home/slow_data/Air_Quality/Himawari/station_aod_v3/L3')
VIIRS_FILE = Path('/home/slow_data/Air_Quality/VIIRS/output/viirs_aod_L2_NOAA20_nearest_raw.csv')
GADM_L1    = Path('/home/work1/projects/Air_Quality/GADM_Vietnam/gadm41_VNM_1.shp')
MASTERDATA = Path('/home/work1/projects/Air_Quality/Masterdata/envisoft_station_map.csv')

# ── Column names ────────────────────────────────────────────────────────────
MODIS_BAND   = 'Optical_Depth_055'
MODIS_COL    = 'AOD_MODIS_055'
HIMA_050_COL = 'AOD_Himawari'
HIMA_055_COL = 'AOD_Himawari_055'
VIIRS_COL    = 'AOD_VIIRS'

HIMA_VARIANTS = [
    (HIMA_050_COL, 'Himawari 0.50 µm (original)',      '#2D7DBB'),
    (HIMA_055_COL, 'Himawari 0.55 µm (AE-corrected)', '#1B4F8A'),
]

# ALL_VARIANTS: all satellites compared against MODIS (Himawari variants + VIIRS)
ALL_VARIANTS = HIMA_VARIANTS + [
    (VIIRS_COL, 'VIIRS (NOAA-20)', '#2E8B57'),
]

# ── Region colours ──────────────────────────────────────────────────────────
REGION_COLORS = {'North': '#E84040', 'Central': '#F5A623', 'South': '#4A90E2'}
REGIONS       = ['North', 'Central', 'South']

# ── Station masterdata: stationName (= CSV filename stem) → (lat, lon) ──────
_meta = pd.read_csv(MASTERDATA)
name_to_coords: dict[str, tuple[float, float]] = dict(
    zip(_meta['stationName'],
        zip(_meta['latitude'].astype(float), _meta['longitude'].astype(float)))
)
print(f'Masterdata loaded    : {len(name_to_coords)} station coords')
print('MODIS dir  :', MODIS_DIR.exists())
print('Himawari dir:', HIMA_DIR.exists())
print('VIIRS file :', VIIRS_FILE.exists())
print('GADM L1    :', GADM_L1.exists())

## 1 · Discover Station Files

In [ ]:
modis_files: dict[str, list[Path]] = {}
for f in MODIS_DIR.glob('*.csv'):
    modis_files.setdefault(f.stem, []).append(f)

hima_files: dict[str, list[Path]] = {}
for f in HIMA_DIR.glob('*.csv'):
    hima_files.setdefault(f.stem, []).append(f)

# Load VIIRS (single file covering all 27 masterdata stations)
viirs_by_station: dict[str, pd.DataFrame] = {}
if VIIRS_FILE.exists():
    _viirs_raw = pd.read_csv(VIIRS_FILE, parse_dates=['datetime'], low_memory=False)
    _viirs_raw['datetime'] = pd.to_datetime(_viirs_raw['datetime'])
    for _sid, _grp in _viirs_raw.groupby('station'):
        viirs_by_station[_sid] = (
            _grp.rename(columns={'datetime': 'timestamp'})
            .set_index('timestamp').sort_index()
        )

common_ids = sorted(set(modis_files) & set(hima_files))
print(f'MODIS stations      : {len(modis_files)}')
print(f'Himawari stations   : {len(hima_files)}')
print(f'VIIRS stations      : {len(viirs_by_station)}')
print(f'Common (MODIS∩Hima) : {len(common_ids)}')
print(f'VIIRS in common_ids : {len(set(common_ids) & set(viirs_by_station))}')

## 2 · Loader Functions

In [ ]:
def load_modis(paths: list[Path]) -> pd.DataFrame:
    frames = [pd.read_csv(p, parse_dates=['timestamp'], low_memory=False) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.dropna(subset=['timestamp']).sort_values('timestamp').set_index('timestamp')
    if 'stationName' in df.columns and 'Name' not in df.columns:
        df = df.rename(columns={'stationName': 'Name'})
    out = df[[c for c in ['Name', 'Latitude', 'Longitude'] if c in df.columns]].copy()
    out[MODIS_COL] = df[MODIS_BAND].astype(float)
    return out


def load_himawari(paths: list[Path]) -> pd.DataFrame:
    frames = [pd.read_csv(p, parse_dates=['timestamp'], low_memory=False) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.dropna(subset=['timestamp']).sort_values('timestamp').set_index('timestamp')
    out = pd.DataFrame({HIMA_050_COL: df['AOT_Merged_center'].astype(float)})
    if 'AE_Merged' in df.columns:
        out['AE_Himawari'] = df['AE_Merged'].astype(float)
    if 'QA_flag_Merged' in df.columns:
        out['QA_Himawari'] = df['QA_flag_Merged']
    return out


print('Loader functions defined ✓')

## 3 · Build Merged Dataset

Same ±1 h window approach as v3. Additionally we track **raw monthly observation counts** per sensor  
per station — used later to diagnose data gaps independent of the co-location filter.

In [ ]:
records       = []
station_dfs   = {}

# Raw monthly observation counts (sensor → station → pd.Series of monthly counts)
modis_raw_monthly: dict[str, pd.Series] = {}
hima_raw_monthly:  dict[str, pd.Series] = {}
viirs_raw_monthly: dict[str, pd.Series] = {}
station_coords:    dict[str, tuple]     = {}   # sid → (lat, lon)

_radius  = pd.Timedelta('1h')
WL_RATIO = 0.55 / 0.50

for sid in common_ids:
    try:
        modis = load_modis(modis_files[sid])
        hima  = load_himawari(hima_files[sid])

        # ── Raw monthly counts for gap analysis ──────────────────────────
        modis_raw_monthly[sid] = modis[MODIS_COL].dropna().resample('ME').count()
        hima_raw_monthly[sid]  = hima[HIMA_050_COL].dropna().resample('ME').count()

        # VIIRS raw monthly counts (empty series for stations not in VIIRS)
        if sid in viirs_by_station:
            viirs_raw_monthly[sid] = (
                viirs_by_station[sid]['aod'].dropna().resample('ME').count()
            )
        else:
            viirs_raw_monthly[sid] = pd.Series(dtype=float)

        # ── Station coordinates from masterdata (keyed by stationName = sid) ──
        station_coords[sid] = name_to_coords.get(sid, (np.nan, np.nan))

        # ── ±1 h merge ────────────────────────────────────────────────────
        hima_aod = hima[HIMA_050_COL]
        has_ae   = 'AE_Himawari' in hima.columns
        has_viirs = sid in viirs_by_station
        viirs_aod_series = viirs_by_station[sid]['aod'] if has_viirs else None

        aod_means, ae_means, viirs_means = [], [], []
        for ts in modis.index:
            t0, t1  = ts - _radius, ts + _radius
            win_aod = hima_aod.loc[t0:t1]
            aod_means.append(win_aod.mean() if len(win_aod) > 0 else np.nan)
            if has_ae:
                win_ae = hima['AE_Himawari'].loc[t0:t1]
                ae_means.append(win_ae.mean() if len(win_ae) > 0 else np.nan)
            if has_viirs:
                win_v = viirs_aod_series.loc[t0:t1]
                viirs_means.append(win_v.mean() if len(win_v) > 0 else np.nan)
            else:
                viirs_means.append(np.nan)

        merged = modis.copy()
        merged[HIMA_050_COL]  = aod_means
        merged['AE_Himawari'] = ae_means if has_ae else np.nan

        ae_col = merged['AE_Himawari']
        merged[HIMA_055_COL] = np.where(
            ae_col.notna(),
            merged[HIMA_050_COL] * WL_RATIO ** (-ae_col),
            merged[HIMA_050_COL]
        )
        merged[VIIRS_COL] = viirs_means

        merged = merged.dropna(subset=[HIMA_050_COL])
        if len(merged) == 0:
            continue

        merged['station_id'] = sid
        station_dfs[sid] = merged
        records.append(merged.reset_index())

    except Exception as e:
        print(f'[WARN] {sid[:40]}: {e}')

all_df = pd.concat(records, ignore_index=True)
all_df['timestamp'] = pd.to_datetime(all_df['timestamp'], utc=True)

# ── Attach lat/lon from masterdata so downstream cells can use them ──────────
all_df['Latitude']  = all_df['station_id'].map({s: v[0] for s, v in station_coords.items()})
all_df['Longitude'] = all_df['station_id'].map({s: v[1] for s, v in station_coords.items()})

n_with_coords = all_df['Latitude'].notna().sum()
n_with_viirs  = all_df[VIIRS_COL].notna().sum()
print(f'Total co-located records : {len(all_df):,}')
print(f'Stations with data       : {all_df["station_id"].nunique()}')
print(f'Records with coords      : {n_with_coords:,} / {len(all_df):,}')
print(f'Records with VIIRS AOD   : {n_with_viirs:,} / {len(all_df):,}')
print(f'Date range               : {all_df["timestamp"].min().date()} → {all_df["timestamp"].max().date()}')

## 4 · Assign Regions via GADM Level-1

We perform a **spatial join** of station coordinates against Vietnam's Level-1 (province) boundaries  
from GADM 4.1. Provinces are then mapped to **North / Central / South** using a lookup dictionary;  
a latitude-based fallback handles any unmatched provinces.

In [ ]:
# ── Load GADM Level-1 ────────────────────────────────────────────────────────
gadm_l1 = gpd.read_file(GADM_L1)
print('GADM CRS :', gadm_l1.crs)
print('Provinces :', len(gadm_l1))
print('Columns   :', gadm_l1.columns.tolist())
gadm_l1[['NAME_1', 'geometry']].head(8)

In [ ]:
# ── Province → region dictionary ────────────────────────────────────────────
# Based on the popular 3-region classification of Vietnam.
# Keys mirror the NAME_1 field in GADM 4.1 (Vietnamese with diacritics).
REGION_MAP: dict[str, str] = {
    # ── NORTH (Miền Bắc) ────────────────────────────────────────────────────
    'Bắc Giang': 'North',  'Bắc Kạn': 'North',   'Bắc Ninh': 'North',
    'Cao Bằng':  'North',  'Điện Biên': 'North',  'Hà Giang': 'North',
    'Hà Nam':    'North',  'Hà Nội':   'North',   'Hà Tĩnh':  'North',
    'Hải Dương': 'North',  'Hải Phòng': 'North',  'Hòa Bình': 'North',
    'Hưng Yên':  'North',  'Lai Châu': 'North',   'Lạng Sơn': 'North',
    'Lào Cai':   'North',  'Nam Định': 'North',   'Nghệ An':  'North',
    'Ninh Bình': 'North',  'Phú Thọ':  'North',   'Quảng Ninh': 'North',
    'Sơn La':    'North',  'Thái Bình': 'North',  'Thái Nguyên': 'North',
    'Thanh Hóa': 'North',  'Tuyên Quang': 'North','Vĩnh Phúc': 'North',
    'Yên Bái':   'North',
    # English / simplified variants sometimes used by GADM
    'Ha Noi': 'North',  'Hanoi': 'North',  'Ho Chi Minh': 'South',
    # ── CENTRAL (Miền Trung) ────────────────────────────────────────────────
    'Bình Định':     'Central',  'Bình Thuận':    'Central',
    'Đà Nẵng':       'Central',  'Đắk Lắk':       'Central',
    'Đắk Nông':      'Central',  'Gia Lai':        'Central',
    'Khánh Hòa':     'Central',  'Kon Tum':        'Central',
    'Lâm Đồng':      'Central',  'Ninh Thuận':     'Central',
    'Phú Yên':       'Central',  'Quảng Bình':     'Central',
    'Quảng Nam':     'Central',  'Quảng Ngãi':     'Central',
    'Quảng Trị':     'Central',  'Thừa Thiên Huế': 'Central',
    # ── SOUTH (Miền Nam) ────────────────────────────────────────────────────
    'An Giang':           'South',  'Bà Rịa - Vũng Tàu': 'South',
    'Bạc Liêu':           'South',  'Bến Tre':             'South',
    'Bình Dương':         'South',  'Bình Phước':          'South',
    'Cà Mau':             'South',  'Cần Thơ':             'South',
    'Đồng Nai':           'South',  'Đồng Tháp':           'South',
    'Hậu Giang':          'South',  'Kiên Giang':          'South',
    'Long An':            'South',  'Sóc Trăng':           'South',
    'Tây Ninh':           'South',  'Tiền Giang':          'South',
    'TP. Hồ Chí Minh':    'South',  'Hồ Chí Minh':         'South',
    'Trà Vinh':           'South',  'Vĩnh Long':           'South',
}


def lat_region_fallback(lat: float) -> str:
    """Fallback latitude-based region assignment."""
    if lat >= 17.5:
        return 'North'
    elif lat >= 11.0:
        return 'Central'
    else:
        return 'South'


print('Region map built ✓')

In [ ]:
# ── Build station GeoDataFrame from masterdata coordinates ───────────────────
coords_df = pd.DataFrame([
    {'station_id': sid, 'Latitude': lat, 'Longitude': lon}
    for sid, (lat, lon) in station_coords.items()
]).dropna(subset=['Latitude', 'Longitude'])

print(f'Stations with coords: {len(coords_df)} / {len(station_coords)}')

stations_gdf = gpd.GeoDataFrame(
    coords_df,
    geometry=gpd.points_from_xy(coords_df['Longitude'], coords_df['Latitude']),
    crs='EPSG:4326'
)

# ── Spatial join: station point → GADM province polygon ─────────────────────
joined = gpd.sjoin(
    stations_gdf,
    gadm_l1[['NAME_1', 'geometry']],
    how='left',
    predicate='within'
)

# ── Assign region ────────────────────────────────────────────────────────────
def assign_region(row) -> str:
    name = str(row.get('NAME_1', ''))
    if name in REGION_MAP:
        return REGION_MAP[name]
    name_low = name.lower()
    for k, v in REGION_MAP.items():
        if k.lower() == name_low:
            return v
    return lat_region_fallback(row['Latitude'])


joined['region'] = joined.apply(assign_region, axis=1)
station_region   = joined.set_index('station_id')['region'].to_dict()
station_province = joined.set_index('station_id')['NAME_1'].to_dict()

# ── Attach region & province to all_df ───────────────────────────────────────
all_df['region']   = all_df['station_id'].map(station_region)
all_df['province'] = all_df['station_id'].map(station_province)

# Fill remaining NaN (stations not in masterdata) with lat-based fallback
mask = all_df['region'].isna()
if mask.any():
    all_df.loc[mask, 'region'] = all_df.loc[mask, 'Latitude'].map(lat_region_fallback)
    print(f'Filled {mask.sum()} rows via lat-based fallback')

print('Stations per region:')
print(all_df.groupby('region')['station_id'].nunique().reindex(REGIONS))
print(f'\nStill unmatched: {all_df["region"].isna().sum()} rows')

In [ ]:
# ── Station map coloured by region ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 9))

gadm_l1.boundary.plot(ax=ax, color='#AAAAAA', linewidth=0.5)

for region, color in REGION_COLORS.items():
    subset = joined[joined['region'] == region]
    ax.scatter(
        subset['Longitude'], subset['Latitude'],
        color=color, s=40, edgecolors='white', linewidths=0.5,
        label=f'{region} ({len(subset)} stations)', zorder=3
    )

ax.set_title('Monitoring Stations by Region\n(GADM Level-1 spatial join)', fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# ── Summary table ────────────────────────────────────────────────────────────
summary = (
    all_df.groupby('region')
    .agg(
        stations=('station_id', 'nunique'),
        records=('station_id', 'count'),
        date_min=('timestamp', 'min'),
        date_max=('timestamp', 'max'),
    )
    .reindex(REGIONS)
)
summary['date_min'] = summary['date_min'].dt.date
summary['date_max'] = summary['date_max'].dt.date
print('\nData summary by region:')
display(summary)

In [ ]:
# ── AERONET stations map (Nghia Do & Bac Lieu) with N/C/S dividers ──────────
AERONET_CSV = '/home/work1/projects/Air_Quality/Masterdata/AERONET_sites.csv'
TARGET_SITES = ['NGHIA_DO', 'BAC_LIEU']  # match in upper-case for robustness

aero = pd.read_csv(AERONET_CSV)
aero = aero[aero['stationName'].str.upper().isin(TARGET_SITES)].copy()

# Per-station colours (consistent with REGION_COLORS palette)
SITE_COLORS = {'NGHIA_DO': '#E84040', 'BAC_LIEU': '#4A90E2'}

fig, ax = plt.subplots(figsize=(7, 9))
gadm_l1.boundary.plot(ax=ax, color='#AAAAAA', linewidth=0.5)

# Region dividers at 16°N (North/Central) and 11.5°N (Central/South)
for y in (16.0, 11.5):
    ax.axhline(y=y, color='black', linestyle='--', linewidth=1.0, zorder=2)

# Stations
for _, row in aero.iterrows():
    key = row['stationName'].upper()
    ax.scatter(
        row['longitude'], row['latitude'],
        color=SITE_COLORS.get(key, 'black'),
        marker='^', s=120, edgecolors='white', linewidths=0.8,
        label=f"{row['stationName']} ({row['latitude']:.2f}°N, {row['longitude']:.2f}°E)",
        zorder=3,
    )

# Region labels on the right margin
xmax = ax.get_xlim()[1]
ax.text(xmax, 18.5, ' North',   ha='left', va='center', fontsize=10, fontweight='bold', alpha=0.75)
ax.text(xmax, 13.75, ' Central', ha='left', va='center', fontsize=10, fontweight='bold', alpha=0.75)
ax.text(xmax, 10.0, ' South',   ha='left', va='center', fontsize=10, fontweight='bold', alpha=0.75)

# ax.set_title('AERONET sites — Nghia Do & Bac Lieu\n'
#              'with North / Central / South dividers (16°N, 11.5°N)',
#              fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(fontsize=10, loc='lower left')
plt.tight_layout()
plt.show()


In [ ]:

# ── Station map coloured by region ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 9))

gadm_l1.boundary.plot(ax=ax, color='#AAAAAA', linewidth=0.5)

# 1. Plot original stations from 'joined' dataframe
for region, color in REGION_COLORS.items():
    subset = joined[joined['region'] == region]
    ax.scatter(
        subset['Longitude'], subset['Latitude'],
        color=color, s=40, edgecolors='white', linewidths=0.5,
        label=f'{region} ({len(subset)} stations)', zorder=3
    )

# --- NEW ADDITIONS START HERE ---

# 2. Load and filter AERONET stations
aeronet_path = '/home/work1/projects/Air_Quality/Masterdata/AERONET_sites.csv'
aeronet_df = pd.read_csv(aeronet_path)

# Filter for specific station names
target_stations = ['Nghia_DO', 'Bac_Lieu']
aeronet_filtered = aeronet_df[aeronet_df['stationName'].isin(target_stations)]

# Plot the filtered AERONET stations
# Note: Ensure your CSV columns are named 'Longitude' and 'Latitude'
ax.scatter(
    aeronet_filtered['Longitude'], aeronet_filtered['Latitude'],
    color='magenta', marker='^', s=100, edgecolors='black', linewidths=1,
    label='AERONET (Nghia_DO & Bac_Lieu)', zorder=4
)

# 3. Add lines to divide North, Centre, and South
# Using axhline to draw horizontal lines across the plot
ax.axhline(y=16.0, color='black', linestyle='--', linewidth=1.2, zorder=2)
ax.axhline(y=11.5, color='black', linestyle='--', linewidth=1.2, zorder=2)

# Optional: Add text labels for the regions on the left side of the plot
xmin = ax.get_xlim()[0]
ax.text(xmin, 16.5, '  NORTH', color='black', fontsize=9, fontweight='bold', alpha=0.7)
ax.text(xmin, 13.5, '  CENTRE', color='black', fontsize=9, fontweight='bold', alpha=0.7)
ax.text(xmin, 10.5, '  SOUTH', color='black', fontsize=9, fontweight='bold', alpha=0.7)

# --- NEW ADDITIONS END HERE ---

ax.set_title('Monitoring Stations by Region\n(GADM Level-1 spatial join)', fontweight='bold')
ax.set_xlabel('Longitude') 
ax.set_ylabel('Latitude')
ax.legend(fontsize=10, loc='upper right') # adjusted location to avoid overlapping lines
plt.tight_layout()
plt.show()

## 5 · Helper — Regression Statistics

In [ ]:
def regression_stats(x: pd.Series, y: pd.Series) -> dict:
    mask = x.notna() & y.notna()
    x, y = x[mask].values, y[mask].values
    if len(x) < 3:
        return dict(n=len(x), r=np.nan, R2=np.nan, slope=np.nan,
                    intercept=np.nan, RMSE=np.nan, MAE=np.nan,
                    Bias=np.nan, RelBias_pct=np.nan)
    slope, intercept, r, _, _ = stats.linregress(x, y)
    rmse    = np.sqrt(mean_squared_error(x, y))
    mae     = mean_absolute_error(x, y)
    bias    = np.mean(y - x)
    relbias = bias / np.mean(x) * 100
    return dict(n=len(x), r=r, R2=r**2, slope=slope, intercept=intercept,
                RMSE=rmse, MAE=mae, Bias=bias, RelBias_pct=relbias)


print('regression_stats defined ✓')

## 6 · Regional Correlation — Scatter Plots

One column per region × one row per Himawari variant.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))

regional_gs: dict[tuple, dict] = {}   # (region, sat_col) → stats

for row_i, (sat_col, sat_label, _) in enumerate(ALL_VARIANTS):
    for col_i, region in enumerate(REGIONS):
        ax     = axes[row_i, col_i]
        color  = REGION_COLORS[region]
        subset = all_df[all_df['region'] == region]
        gs     = regression_stats(subset[MODIS_COL], subset[sat_col])
        regional_gs[(region, sat_col)] = gs

        valid = subset[[MODIS_COL, sat_col]].dropna()
        if len(valid) == 0:
            ax.set_title(f'{region}  —  {sat_label}\n(no data)', fontweight='bold', fontsize=9)
            continue

        ax.scatter(
            valid[MODIS_COL], valid[sat_col],
            c=color, edgecolors='white', linewidths=0.4, s=30, alpha=0.70
        )

        q99  = max(valid[MODIS_COL].quantile(0.99), valid[sat_col].quantile(0.99))
        lims = [0, q99 * 1.05] if q99 > 0 else [0, 1]
        xfit = np.linspace(*lims, 200)
        ax.plot(lims, lims, 'k--', lw=1.2, label='1:1')
        if not np.isnan(gs['slope']):
            ax.plot(xfit, gs['slope']*xfit + gs['intercept'], 'b-', lw=2,
                    label=f"y={gs['slope']:.3f}x{gs['intercept']:+.3f}")
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('MODIS 0.55 µm AOD', fontsize=9)
        ax.set_ylabel(sat_label, fontsize=9)
        ax.set_title(f'{region}  —  {sat_label}\n(n={gs["n"]:,})', fontweight='bold', fontsize=9)

        txt = (f"R²={gs['R2']:.3f}  r={gs['r']:.3f}\n"
               f"RMSE={gs['RMSE']:.3f}  MAE={gs['MAE']:.3f}\n"
               f"Bias={gs['Bias']:+.3f}  ({gs['RelBias_pct']:+.1f}%)")
        ax.text(0.03, 0.97, txt, transform=ax.transAxes,
                va='top', ha='left', fontsize=8, fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.9))
        ax.legend(fontsize=8, loc='lower right')

plt.suptitle(
    'MODIS 0.55 µm vs Satellite AOD — Regional Correlation\n'
    'Row 1: Himawari 0.50 µm (original)  |  Row 2: Himawari 0.55 µm (AE-corrected)  |  Row 3: VIIRS (NOAA-20)',
    fontweight='bold', fontsize=12
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 7 · Regional Statistics Summary Table

In [ ]:
rows = []
for region in REGIONS:
    for sat_col, sat_label, _ in ALL_VARIANTS:
        gs = regional_gs[(region, sat_col)]
        rows.append({'Region': region, 'Satellite': sat_label, **gs})

reg_stats_df = pd.DataFrame(rows).set_index(['Region', 'Satellite'])
display(reg_stats_df[['n', 'R2', 'r', 'RMSE', 'MAE', 'Bias', 'RelBias_pct', 'slope', 'intercept']].round(4))

## 8 · Per-Station R² Distribution by Region

In [ ]:
# ── Compute per-station R² for each satellite variant ─────────────────────────
per_station_rows = []
for sid, df_s in station_dfs.items():
    if len(df_s) < 5:
        continue
    region = station_region.get(sid, lat_region_fallback(station_coords[sid][0]))
    for sat_col, sat_label, _ in ALL_VARIANTS:
        s = regression_stats(df_s[MODIS_COL], df_s[sat_col])
        per_station_rows.append({
            'station_id': sid, 'region': region,
            'sat_variant': sat_label, **s
        })

per_station_df = pd.DataFrame(per_station_rows)

# ── Box plots (one panel per satellite variant) ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 5), sharey=True)

for ax, (sat_col, sat_label, _) in zip(axes, ALL_VARIANTS):
    subset = per_station_df[per_station_df['sat_variant'] == sat_label]

    data_by_region = [
        subset.loc[subset['region'] == r, 'R2'].dropna().values
        for r in REGIONS
    ]
    bp = ax.boxplot(
        data_by_region, labels=REGIONS, patch_artist=True,
        medianprops=dict(color='black', lw=2)
    )
    for patch, region in zip(bp['boxes'], REGIONS):
        patch.set_facecolor(REGION_COLORS[region])
        patch.set_alpha(0.75)

    for i, (region, data) in enumerate(zip(REGIONS, data_by_region), start=1):
        ax.scatter(
            np.random.normal(i, 0.07, size=len(data)), data,
            color=REGION_COLORS[region], alpha=0.55, s=20, zorder=3
        )

    ax.set_ylabel('Per-station R²')
    ax.set_title(f'R² distribution\nvs {sat_label}', fontweight='bold')
    ax.axhline(0.7, color='gray', lw=1.2, linestyle='--', label='R²=0.7 threshold')
    ax.legend(fontsize=8)

plt.suptitle('Per-Station R² by Region — MODIS 0.55 µm vs Satellite AOD\n'
             'Left: Himawari 0.50 µm  |  Centre: Himawari 0.55 µm (AE-corrected)  |  Right: VIIRS (NOAA-20)',
             fontweight='bold', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# ── Median R² per region ─────────────────────────────────────────────────────
print('Median per-station R²:')
print(
    per_station_df
    .groupby(['region', 'sat_variant'])['R2']
    .mean()
    .unstack('sat_variant')
    .reindex(REGIONS)
    .round(3)
)

## 9 · Monthly Correlation Trend by Region

In [ ]:
all_df['month'] = all_df['timestamp'].dt.month
month_names     = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(21, 5), sharey=False)

for ax, (sat_col, sat_label, _) in zip(axes, ALL_VARIANTS):
    for region in REGIONS:
        color  = REGION_COLORS[region]
        subset = all_df[all_df['region'] == region]
        monthly_r2 = []
        months     = []
        for m, grp in subset.groupby('month'):
            valid = grp[[MODIS_COL, sat_col]].dropna()
            if len(valid) >= 10:
                r, _ = stats.pearsonr(valid[MODIS_COL], valid[sat_col])
                monthly_r2.append(r**2)
                months.append(m)
        ax.plot(months, monthly_r2, marker='o', color=color,
                lw=2, ms=7, label=region)

    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names)
    ax.set_ylabel('R²'); ax.set_ylim(0, 1)
    ax.set_title(f'Monthly R² by Region\nvs {sat_label}', fontweight='bold')
    ax.axhline(0.7, color='gray', lw=1, linestyle='--', alpha=0.6)
    ax.legend(fontsize=9)

plt.suptitle('Seasonal Variation of MODIS Agreement by Region\n'
             'Left: Himawari 0.50 µm  |  Centre: Himawari 0.55 µm  |  Right: VIIRS (NOAA-20)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 10 · Missing Data Time Series

Using the **raw per-station observation counts** collected during the build step (before the ±1 h co-location  
filter), we compute two metrics per month:

- **Station coverage %**: fraction of stations that have ≥1 observation that month  
- **Total observations**: summed raw observation count across all stations

If MODIS and Himawari gaps coincide → common meteorological driver (e.g. persistent cloud cover).  
If only one sensor drops → sensor-specific issue (orbital gap, product outage, processing failure).

In [ ]:
# ── Aggregate raw monthly counts across all stations ─────────────────────────
def monthly_coverage_df(
    raw_monthly: dict[str, pd.Series],
    sid_region: dict[str, str]
) -> pd.DataFrame:
    """
    Returns a long DataFrame with columns:
      month_end, station_id, region, obs_count
    """
    rows = []
    for sid, series in raw_monthly.items():
        if series.empty:
            continue
        region = sid_region.get(sid, lat_region_fallback(station_coords[sid][0]))
        for month_end, count in series.items():
            rows.append({
                'month_end':  month_end,
                'station_id': sid,
                'region':     region,
                'obs_count':  count,
            })
    return pd.DataFrame(rows)


modis_cov_long = monthly_coverage_df(modis_raw_monthly, station_region)
hima_cov_long  = monthly_coverage_df(hima_raw_monthly,  station_region)
viirs_cov_long = monthly_coverage_df(viirs_raw_monthly, station_region)

# ── Global monthly aggregates ─────────────────────────────────────────────────
def global_monthly(cov_long: pd.DataFrame) -> pd.DataFrame:
    if cov_long.empty:
        return pd.DataFrame()
    g = cov_long.groupby('month_end')
    total_stations = cov_long['station_id'].nunique()
    return pd.DataFrame({
        'total_obs':          g['obs_count'].sum(),
        'stations_with_data': g.apply(lambda x: (x['obs_count'] > 0).sum()),
        'coverage_pct':       g.apply(lambda x: 100 * (x['obs_count'] > 0).sum() / total_stations),
    })

modis_global = global_monthly(modis_cov_long)
hima_global  = global_monthly(hima_cov_long)
viirs_global = global_monthly(viirs_cov_long)

print('Monthly coverage computed ✓')
print(f'MODIS  date range: {modis_global.index.min().date()} → {modis_global.index.max().date()}')
print(f'Hima   date range: {hima_global.index.min().date()} → {hima_global.index.max().date()}')
print(f'VIIRS  date range: {viirs_global.index.min().date()} → {viirs_global.index.max().date()}')
print(f'VIIRS  stations  : {viirs_cov_long["station_id"].nunique()} (masterdata stations only)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# ── Panel 1: station coverage % ───────────────────────────────────────────────
ax = axes[0]
ax.plot(modis_global.index, modis_global['coverage_pct'],
        color='#C4782A', lw=2, marker='o', ms=4, label='MODIS')
ax.plot(hima_global.index,  hima_global['coverage_pct'],
        color='#2D7DBB', lw=2, marker='s', ms=4, label='Himawari')
ax.plot(viirs_global.index, viirs_global['coverage_pct'],
        color='#2E8B57', lw=2, marker='^', ms=4, label='VIIRS (NOAA-20, 27 stations)')
ax.fill_between(modis_global.index, modis_global['coverage_pct'], alpha=0.10, color='#C4782A')
ax.fill_between(hima_global.index,  hima_global['coverage_pct'],  alpha=0.10, color='#2D7DBB')
ax.fill_between(viirs_global.index, viirs_global['coverage_pct'], alpha=0.10, color='#2E8B57')
ax.set_ylabel('Stations with data (%)', fontsize=11)
ax.set_title('Monthly Station Coverage — What fraction of stations reported data?',
             fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim(0, 105)
ax.axhline(50, color='gray', lw=1, linestyle='--', alpha=0.5)

# ── Panel 2: total observations ───────────────────────────────────────────────
ax2 = axes[1]
x_modis = mdates.date2num(modis_global.index)
x_hima  = mdates.date2num(hima_global.index)
x_viirs = mdates.date2num(viirs_global.index)
w = 18
ax2.bar(x_modis - w, modis_global['total_obs'],
        width=w, color='#C4782A', alpha=0.8, edgecolor='black', linewidth=0.5, label='MODIS')
ax2.bar(x_hima,      hima_global['total_obs'],
        width=w, color='#2D7DBB', alpha=0.8, edgecolor='black', linewidth=0.5, label='Himawari')
ax2.bar(x_viirs + w, viirs_global['total_obs'],
        width=w, color='#2E8B57', alpha=0.8, edgecolor='black', linewidth=0.5,
        label='VIIRS (NOAA-20)')
ax2.set_ylabel('Total observations (all stations)', fontsize=11)
ax2.set_title('Monthly Total Observations', fontweight='bold')
ax2.legend(fontsize=10)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Data Availability Time Series — MODIS, Himawari & VIIRS (All Stations)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Create the figure and axes explicitly to prevent the "0 Axes" empty figure issue
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6))
ax2 = axes[1]

# Filter data to only include dates from 2022-09 onwards
modis_subset = modis_global.loc['2022-09':]
hima_subset = hima_global.loc['2022-09':]
viirs_subset = viirs_global.loc['2022-09':]

# 2. IMPROVED BAR AESTHETICS
# Added 'edgecolor' and 'linewidth' to make the bars crisp, especially for print.
# Increased alpha slightly to prevent washed-out colors on paper.

# Convert DatetimeIndex to matplotlib date numbers to prevent 'axis units' errors.
x_modis = mdates.date2num(modis_subset.index)
x_hima = mdates.date2num(hima_subset.index)
x_viirs = mdates.date2num(viirs_subset.index)

# Use width=9 and offset the bars so all three fit within a single month without overlapping
w = 9

ax2.bar(x_modis - w, modis_subset['total_obs'],
        width=w, color='#C4782A', alpha=0.85, edgecolor='black', linewidth=0.6, label='MODIS')

ax2.bar(x_hima, hima_subset['total_obs'],
        width=w, color='#2D7DBB', alpha=0.85, edgecolor='black', linewidth=0.6, label='Himawari')

ax2.bar(x_viirs + w, viirs_subset['total_obs'],
        width=w, color='#2E8B57', alpha=0.85, edgecolor='black', linewidth=0.6, label='VIIRS (NOAA-20)')

# 3. CLEAN LABELS & TITLES
ax2.set_ylabel('Total observations', weight='medium')
# Note: In papers, titles are often omitted in favor of figure captions. 
# If you keep it, standardizing to a format like "(b)" is common.
ax2.set_title('(b) Monthly Total Observations', loc='left', fontweight='bold', pad=10)

# Legend without a box frame looks cleaner in documents.
# Using ncol=3 helps the legend spread horizontally instead of blocking vertical data.
ax2.legend(frameon=False, loc='upper right', ncol=3)

# 4. SUBTLE GRID & BORDERS (DESPINING)
# Only keep the horizontal grid to help read values, make it dashed and faint
ax2.yaxis.grid(True, linestyle='--', alpha=0.5)
# Remove the top and right bounding box lines
sns.despine(ax=ax2) 

# 5. X-AXIS FORMATTING
# Fixed your snippet to use ax2 instead of ax
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

# Use 45 degrees with 'anchor' alignment so the text points exactly at the tick mark
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right', rotation_mode='anchor')

# Always call tight_layout before saving to prevent cut-off labels
plt.tight_layout()

# Display the plot (prevents just printing the figure object string in Jupyter/console)
plt.show()

## 11 · Missing Data Time Series by Region

Same analysis broken down into **North / Central / South** subplots to reveal whether gaps  
are spatially uniform (e.g. cloud events that blanket the whole country) or regionally specific.

In [ ]:
def regional_monthly(cov_long: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    if cov_long.empty:
        return out
    for region in REGIONS:
        sub = cov_long[cov_long['region'] == region]
        n_stations = sub['station_id'].nunique()
        if n_stations == 0:
            continue
        g = sub.groupby('month_end')
        out[region] = pd.DataFrame({
            'total_obs':          g['obs_count'].sum(),
            'stations_with_data': g.apply(lambda x: (x['obs_count'] > 0).sum()),
            'coverage_pct':       g.apply(
                lambda x: 100 * (x['obs_count'] > 0).sum() / n_stations
            ),
            'n_stations': n_stations,
        })
    return out


modis_regional = regional_monthly(modis_cov_long)
hima_regional  = regional_monthly(hima_cov_long)
viirs_regional = regional_monthly(viirs_cov_long)

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)

for ax, region in zip(axes, REGIONS):
    color = REGION_COLORS[region]
    m_df  = modis_regional.get(region, pd.DataFrame())
    h_df  = hima_regional.get(region,  pd.DataFrame())
    v_df  = viirs_regional.get(region, pd.DataFrame())

    if not m_df.empty:
        ax.plot(m_df.index, m_df['coverage_pct'],
                color='#C4782A', lw=2, marker='o', ms=4, label='MODIS')
        ax.fill_between(m_df.index, m_df['coverage_pct'],
                        color='#C4782A', alpha=0.12)
    if not h_df.empty:
        ax.plot(h_df.index, h_df['coverage_pct'],
                color='#2D7DBB', lw=2, marker='s', ms=4, label='Himawari')
        ax.fill_between(h_df.index, h_df['coverage_pct'],
                        color='#2D7DBB', alpha=0.12)
    if not v_df.empty:
        n_v = v_df['n_stations'].iloc[0]
        ax.plot(v_df.index, v_df['coverage_pct'],
                color='#2E8B57', lw=2, marker='^', ms=4,
                label=f'VIIRS (NOAA-20, {n_v} stn)')
        ax.fill_between(v_df.index, v_df['coverage_pct'],
                        color='#2E8B57', alpha=0.12)

    n_st = m_df['n_stations'].iloc[0] if not m_df.empty else 0
    ax.set_title(f'{region}  ({n_st} MODIS/Himawari stations)', fontweight='bold',
                 fontsize=11, color=color)
    ax.set_ylabel('Station coverage (%)')
    ax.set_ylim(0, 105)
    ax.axhline(50, color='gray', lw=1, linestyle='--', alpha=0.5)
    ax.legend(fontsize=9, loc='upper right')

    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Monthly Station Coverage by Region — MODIS, Himawari & VIIRS',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 12 · Coincident Gap Heatmap

For each month × region we flag whether coverage was **low (< 30%)** for MODIS, Himawari, both, or neither.  
Cells where **both** sensors drop simultaneously are the most interesting — they suggest a shared cause.

In [ ]:
LOW_COVERAGE_THRESHOLD = 30   # % — below this = "low coverage" month

from matplotlib.colors import ListedColormap

# ── Count how many sensors have low coverage each month (0-3) ─────────────────
# 0 = all sensors OK  |  1 = one sensor low  |  2 = two low  |  3 = all low
gap_cmap  = ListedColormap(['#DDEEDD', '#F5A623', '#E06010', '#CC2222'])
gap_ticks = [0, 1, 2, 3]
gap_labels = ['All OK', '1 sensor low', '2 sensors low', 'All 3 low']

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for ax, region in zip(axes, REGIONS):
    m_df = modis_regional.get(region, pd.DataFrame())
    h_df = hima_regional.get(region,  pd.DataFrame())
    v_df = viirs_regional.get(region, pd.DataFrame())

    if m_df.empty or h_df.empty:
        ax.set_title(f'{region}\n(no data)')
        continue

    common_idx = m_df.index.union(h_df.index)
    if not v_df.empty:
        common_idx = common_idx.union(v_df.index)

    m_cov = m_df['coverage_pct'].reindex(common_idx)
    h_cov = h_df['coverage_pct'].reindex(common_idx)
    v_cov = v_df['coverage_pct'].reindex(common_idx) if not v_df.empty else pd.Series(
        np.nan, index=common_idx
    )

    # Count sensors with low coverage; treat NaN VIIRS as "not low" (no data = no failure claim)
    m_low = (m_cov < LOW_COVERAGE_THRESHOLD).astype(int)
    h_low = (h_cov < LOW_COVERAGE_THRESHOLD).astype(int)
    v_low = v_cov.notna() & (v_cov < LOW_COVERAGE_THRESHOLD)
    v_low = v_low.astype(int)
    gap_code = m_low + h_low + v_low

    # Year × month pivot
    gap_df = pd.DataFrame({'gap': gap_code, 'date': common_idx})
    gap_df['year']  = gap_df['date'].dt.year
    gap_df['month'] = gap_df['date'].dt.month
    pivot = gap_df.pivot(index='year', columns='month', values='gap')
    pivot.columns = [['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec'][m-1] for m in pivot.columns]

    im = ax.imshow(pivot.values.astype(float), aspect='auto',
                   cmap=gap_cmap, vmin=-0.5, vmax=3.5, interpolation='none')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index.astype(str), fontsize=9)
    n_viirs = v_df['n_stations'].iloc[0] if not v_df.empty else 0
    ax.set_title(f'{region}\n(VIIRS: {n_viirs} stations, threshold: <{LOW_COVERAGE_THRESHOLD}%)',
                 fontweight='bold', fontsize=10, color=REGION_COLORS[region])

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#DDEEDD', label='All sensors OK'),
    mpatches.Patch(facecolor='#F5A623', label='1 sensor low'),
    mpatches.Patch(facecolor='#E06010', label='2 sensors low'),
    mpatches.Patch(facecolor='#CC2222', label='All 3 sensors low'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.03))

plt.suptitle(
    f'Multi-Sensor Data Gap Calendar (coverage < {LOW_COVERAGE_THRESHOLD}%)\n'
    'Count of MODIS, Himawari & VIIRS sensors with low coverage each month',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.show()

## 13 · Coverage Correlation — Are Sensor Gaps Synchronised?

Scatter of monthly coverage % (MODIS vs Himawari, MODIS vs VIIRS) per region.  
Points near the top-right mean **both sensors had good coverage**; bottom-left means **both were poor**.  
The Pearson r tells us how tightly the gap timing is coupled across sensors.

In [ ]:
sat_pairs = [
    (modis_regional, hima_regional,  'MODIS coverage (%)', 'Himawari coverage (%)',    '#2D7DBB'),
    (modis_regional, viirs_regional, 'MODIS coverage (%)', 'VIIRS (NOAA-20) coverage (%)', '#2E8B57'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row_i, (ref_reg, cmp_reg, xlabel, ylabel, cmp_color) in enumerate(sat_pairs):
    for col_i, region in enumerate(REGIONS):
        ax    = axes[row_i, col_i]
        color = REGION_COLORS[region]
        r_df  = ref_reg.get(region, pd.DataFrame())
        c_df  = cmp_reg.get(region, pd.DataFrame())

        if r_df.empty or c_df.empty:
            ax.set_title(f'{region}\n(insufficient data)')
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            continue

        common_idx = r_df.index.intersection(c_df.index)
        x = r_df.loc[common_idx, 'coverage_pct']
        y = c_df.loc[common_idx, 'coverage_pct']

        # Drop months where either series is NaN
        valid = x.notna() & y.notna()
        x, y = x[valid], y[valid]

        ax.scatter(x, y, color=color, edgecolors='white', s=50, alpha=0.8, zorder=3)

        if len(x) >= 3:
            r, p = stats.pearsonr(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            slope, intercept, *_ = stats.linregress(x, y)
            ax.plot(xfit, slope*xfit + intercept, 'k--', lw=1.5)
            ax.text(0.05, 0.95, f'r = {r:.2f}  (p={p:.3f})',
                    transform=ax.transAxes, va='top', fontsize=10,
                    bbox=dict(boxstyle='round', fc='white', alpha=0.9))

        ax.set_xlim(0, 105); ax.set_ylim(0, 105)
        ax.plot([0, 105], [0, 105], 'gray', lw=1, linestyle=':')
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'{region}\nn={len(x)} months', fontweight='bold', color=color)

row_labels = ['MODIS vs Himawari', 'MODIS vs VIIRS (NOAA-20)']
for row_i, label in enumerate(row_labels):
    axes[row_i, 0].annotate(label, xy=(0, 0.5), xytext=(-axes[row_i, 0].yaxis.labelpad - 30, 0),
                             xycoords='axes fraction', textcoords='offset points',
                             size=11, ha='right', va='center', fontweight='bold', rotation=90)

plt.suptitle('Monthly Coverage Correlation by Region\n'
             'Row 1: MODIS vs Himawari  |  Row 2: MODIS vs VIIRS (NOAA-20)\n'
             'Each point = one calendar month',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 14 · MODIS AOD vs PM2.5 — Regional Correlation

We load PM2.5 from `historical_full_v2` (same source as `EDA_MODIS_AQI.ipynb`) and merge it with  
MODIS AOD using a **±1 h nearest-timestamp** join.  Stations are assigned to North / Central / South  
via the `station_region` dictionary already built in Section 4.

**AOD column used:** `Optical_Depth_055` (same band as the MODIS–Himawari comparison above).  
**PM2.5 filter:** records with PM2.5 ≥ 0 and < 500 µg/m³ only (sentinel/invalid values removed).

In [ ]:
AQ_DIR      = '/home/slow_data/Air_Quality/historical_full_v2'
ERA5_DIR    = '/home/slow_data/Air_Quality/weather'

# F1 filter & physics-correction constants (same as EDA_MODIS_AQI.ipynb)
FINE_MODE_THRESH = 0.5    # F1: FineModeFraction ≥ 0.5
GAMMA            = 0.6    # hygroscopic growth exponent
PBLH_MIN         = 50.0   # minimum PBLH clip (m)
RANSAC_FRAC      = 0.80   # fraction of inliers for RANSAC
PM25_MAX         = 500    # sanity cap on PM2.5


def _clean_ts_df(df, ts_col):
    """Deduplicate a DataFrame on ts_col, keeping the row with most non-null fields."""
    df['_valid'] = df.notnull().sum(axis=1)
    return (df.sort_values([ts_col, '_valid'], ascending=[True, False])
              .drop_duplicates(subset=ts_col, keep='first')
              .drop(columns=['_valid'])
              .sort_values(ts_col)
              .reset_index(drop=True))


def load_aod_aqi_era5(aod_stem: str) -> pd.DataFrame:
    """
    Load one station's MODIS AOD + PM2.5 + ERA5 (RH, PBLH), merge all on timestamp.
    Returns DataFrame with AOD, PM2.5, RH, PBLH, FineModeFraction columns.
    """
    aod_path = MODIS_DIR / f'{aod_stem}.csv'
    if not aod_path.exists():
        return pd.DataFrame()

    df_aod = pd.read_csv(aod_path, parse_dates=['timestamp'], low_memory=False)
    df_aod['timestamp'] = pd.to_datetime(df_aod['timestamp'])
    df_aod = _clean_ts_df(df_aod, 'timestamp')

    # AQI file: ': ' → ' '
    aq_stem  = aod_stem.replace(': ', ' ')
    aq_path  = os.path.join(AQ_DIR, f'{aq_stem}.csv')
    era5_path = os.path.join(ERA5_DIR, f'weather_{aq_stem}.csv')

    if not os.path.isfile(aq_path):
        return pd.DataFrame()

    df_aq = pd.read_csv(aq_path, parse_dates=['Timestamp'], low_memory=False)
    df_aq = df_aq.rename(columns={'Timestamp': 'aq_ts'})
    df_aq = df_aq.replace([-9999, -999, 9999], np.nan)
    if 'PM2.5' in df_aq.columns:
        df_aq.loc[~df_aq['PM2.5'].between(0, PM25_MAX), 'PM2.5'] = np.nan
    df_aq = _clean_ts_df(df_aq, 'aq_ts')

    # Merge AOD ← PM2.5
    merged = pd.merge_asof(
        df_aod.sort_values('timestamp'),
        df_aq[['aq_ts', 'PM2.5']],
        left_on='timestamp', right_on='aq_ts',
        direction='nearest', tolerance=pd.Timedelta('1h'),
    )

    # Merge ← ERA5
    if os.path.isfile(era5_path):
        df_era5 = pd.read_csv(era5_path, parse_dates=['Timestamp'], low_memory=False)
        df_era5 = df_era5.rename(columns={'Timestamp': 'era5_ts', 'Humidity': 'RH'})
        df_era5 = df_era5.replace([-9999, -999, 9999], np.nan)
        if 'RH'   in df_era5.columns: df_era5.loc[~df_era5['RH'].between(0, 100), 'RH'] = np.nan
        if 'PBLH' in df_era5.columns: df_era5.loc[df_era5['PBLH'] < 0, 'PBLH'] = np.nan
        df_era5 = _clean_ts_df(df_era5, 'era5_ts')
        merged = pd.merge_asof(
            merged.sort_values('timestamp'),
            df_era5[['era5_ts', 'RH', 'PBLH']],
            left_on='timestamp', right_on='era5_ts',
            direction='nearest', tolerance=pd.Timedelta('3h'),
        )

    merged['station_id'] = aod_stem
    return merged


# ── Load all stations ─────────────────────────────────────────────────────────
aqi_parts = []
for sid in common_ids:
    df = load_aod_aqi_era5(sid)
    if not df.empty:
        aqi_parts.append(df)

aqi_raw = pd.concat(aqi_parts, ignore_index=True)
aqi_raw['timestamp'] = pd.to_datetime(aqi_raw['timestamp'])
aqi_raw['PM2.5']     = pd.to_numeric(aqi_raw['PM2.5'], errors='coerce')

# ── F1 filter: FineModeFraction ≥ 0.5 ────────────────────────────────────────
aqi_f1 = aqi_raw[
    aqi_raw['FineModeFraction'].notna() &
    (aqi_raw['FineModeFraction'] >= FINE_MODE_THRESH) &
    aqi_raw['PM2.5'].notna() &
    aqi_raw['Optical_Depth_055'].notna()
].copy()

# ── Physics correction: AOD_corr = AOD × (1 − RH/100)^γ / PBLH ──────────────
aqi_f1['PBLH'] = aqi_f1['PBLH'].clip(lower=PBLH_MIN)
has_era5 = aqi_f1['RH'].notna() & aqi_f1['PBLH'].notna()
aqi_f1['AOD_055_corr'] = np.where(
    has_era5,
    aqi_f1['Optical_Depth_055'] * (1 - aqi_f1['RH'] / 100) ** GAMMA / aqi_f1['PBLH'],
    np.nan
)
aqi_f1 = aqi_f1.dropna(subset=['AOD_055_corr'])

# ── Attach region ─────────────────────────────────────────────────────────────
aqi_f1['region'] = aqi_f1['station_id'].map(station_region)
mask_nr = aqi_f1['region'].isna()
if mask_nr.any():
    _lat_map = {s: v[0] for s, v in station_coords.items()}
    aqi_f1.loc[mask_nr, 'region'] = (
        aqi_f1.loc[mask_nr, 'station_id'].map(_lat_map).map(lat_region_fallback)
    )

print(f'After F1 + physics correction:')
print(f'  Records  : {len(aqi_f1):,}')
print(f'  Stations : {aqi_f1["station_id"].nunique()}')
print(f'  PM2.5    : {aqi_f1["PM2.5"].min():.1f} – {aqi_f1["PM2.5"].max():.1f} µg/m³')
print(f'  AOD_corr : {aqi_f1["AOD_055_corr"].min():.5f} – {aqi_f1["AOD_055_corr"].max():.5f}')
print('\nPer region:')
print(aqi_f1.groupby('region').agg(
    records=('PM2.5','count'), stations=('station_id','nunique')
).reindex(REGIONS))

## 15 · Regional Scatter — MODIS AOD 0.55 µm vs PM2.5

In [ ]:
def fit_ransac(x: np.ndarray, y: np.ndarray, frac: float = RANSAC_FRAC) -> dict:
    """RANSAC fit; returns inlier_mask, predicted y, R² (inliers), MAE (inliers), slope, intercept."""
    X = x.reshape(-1, 1)
    min_s = max(2, int(len(x) * frac))
    ransac = RANSACRegressor(estimator=LinearRegression(),
                             min_samples=min_s, random_state=42).fit(X, y)
    mask   = ransac.inlier_mask_
    y_pred = ransac.predict(X)
    r2     = r2_score(y[mask], y_pred[mask])
    mae    = mean_absolute_error(y[mask], y_pred[mask])
    return dict(
        inlier_mask=mask, y_pred=y_pred,
        r2=r2, mae=mae,
        slope=float(ransac.estimator_.coef_[0]),
        intercept=float(ransac.estimator_.intercept_),
    )


AOD_COL_CORR = 'AOD_055_corr'

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
aqi_regional_gs: dict[str, dict] = {}

for ax, region in zip(axes, REGIONS):
    color  = REGION_COLORS[region]
    subset = aqi_f1[aqi_f1['region'] == region][[AOD_COL_CORR, 'PM2.5']].dropna()

    if len(subset) < 5:
        ax.set_title(f'{region}\n(insufficient data)')
        aqi_regional_gs[region] = {}
        continue

    x = subset[AOD_COL_CORR].values
    y = subset['PM2.5'].values
    fit = fit_ransac(x, y)
    aqi_regional_gs[region] = {**fit, 'n': len(x), 'n_inliers': fit['inlier_mask'].sum()}

    # Outliers first (below inliers in z-order)
    ax.scatter(x[~fit['inlier_mask']], y[~fit['inlier_mask']],
               color='lightcoral', alpha=0.4, s=15, label='RANSAC outliers')
    ax.scatter(x[fit['inlier_mask']],  y[fit['inlier_mask']],
               color=color, edgecolors='white', linewidths=0.3,
               alpha=0.65, s=20, label='RANSAC inliers')

    xs = np.linspace(x.min(), np.percentile(x, 99), 200)
    ax.plot(xs, fit['slope'] * xs + fit['intercept'],
            color='navy', lw=2.5,
            label=f"y={fit['slope']:.0f}x{fit['intercept']:+.1f}")

    txt = (f"n={len(x):,}  inliers={fit['inlier_mask'].sum():,}\n"
           f"R²={fit['r2']:.3f}  MAE={fit['mae']:.2f} µg/m³\n"
           f"slope={fit['slope']:.1f}")
    ax.text(0.03, 0.97, txt, transform=ax.transAxes,
            va='top', ha='left', fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.92))
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlabel('MODIS AOD 0.55 µm (physics-corrected)', fontsize=9)
    ax.set_ylabel('PM2.5 (µg/m³)', fontsize=10)
    ax.set_title(region, fontweight='bold', fontsize=12, color=color)
    ax.set_xlim(left=0); ax.set_ylim(bottom=0)

plt.suptitle(
    'MODIS AOD 0.55 µm (F1 + Physics Corrected) vs PM2.5 — RANSAC by Region\n'
    r'$\mathrm{AOD_{corr}} = \mathrm{AOD} \times (1 - \mathrm{RH}/100)^{0.6} \;/\; \mathrm{PBLH}$',
    fontweight='bold', fontsize=12
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 16 · Regional AOD–PM2.5 Statistics Table & Per-Station R² Distribution

In [ ]:
# ── Regional summary table ────────────────────────────────────────────────────
aqi_stats_rows = [
    {'Region': r,
     'n': aqi_regional_gs[r].get('n', 0),
     'n_inliers': aqi_regional_gs[r].get('n_inliers', 0),
     'RANSAC R²': round(aqi_regional_gs[r].get('r2', np.nan), 4),
     'RANSAC MAE': round(aqi_regional_gs[r].get('mae', np.nan), 3),
     'slope': round(aqi_regional_gs[r].get('slope', np.nan), 2),
     'intercept': round(aqi_regional_gs[r].get('intercept', np.nan), 2)}
    for r in REGIONS if aqi_regional_gs.get(r)
]
print('Regional MODIS AOD (F1, physics-corrected) – PM2.5  [RANSAC]:')
display(pd.DataFrame(aqi_stats_rows).set_index('Region'))

# ── Per-station RANSAC R² ─────────────────────────────────────────────────────
per_stn_aqi = []
for sid, grp in aqi_f1.groupby('station_id'):
    sub = grp[[AOD_COL_CORR, 'PM2.5']].dropna()
    if len(sub) < 5:
        continue
    region = station_region.get(sid, lat_region_fallback(station_coords[sid][0]))
    try:
        fit = fit_ransac(sub[AOD_COL_CORR].values, sub['PM2.5'].values)
        per_stn_aqi.append({'station_id': sid, 'region': region,
                            'R2': fit['r2'], 'n': len(sub),
                            'n_inliers': fit['inlier_mask'].sum()})
    except Exception:
        pass

per_stn_aqi_df = pd.DataFrame(per_stn_aqi)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box + jitter
ax = axes[0]
data_by_region = [
    per_stn_aqi_df.loc[per_stn_aqi_df['region'] == r, 'R2'].dropna().values
    for r in REGIONS
]
bp = ax.boxplot(data_by_region, labels=REGIONS, patch_artist=True,
                medianprops=dict(color='black', lw=2))
for patch, region in zip(bp['boxes'], REGIONS):
    patch.set_facecolor(REGION_COLORS[region]); patch.set_alpha(0.75)
for i, (region, data) in enumerate(zip(REGIONS, data_by_region), start=1):
    ax.scatter(np.random.normal(i, 0.07, size=len(data)), data,
               color=REGION_COLORS[region], alpha=0.55, s=20, zorder=3)
ax.set_ylabel('Per-station RANSAC R²  (AOD_corr vs PM2.5)')
ax.set_title('Per-Station RANSAC R² by Region\n(F1, Physics-Corrected MODIS AOD vs PM2.5)',
             fontweight='bold')
ax.axhline(0.5, color='gray', lw=1.2, linestyle='--', label='R²=0.5 guide')
ax.legend(fontsize=8)

# Median bar
ax2 = axes[1]
med_r2 = per_stn_aqi_df.groupby('region')['R2'].median().reindex(REGIONS).fillna(0)
bars = ax2.bar(REGIONS, med_r2.values,
               color=[REGION_COLORS[r] for r in REGIONS],
               edgecolor='white', linewidth=1.2, alpha=0.85)
for bar, val in zip(bars, med_r2.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Median per-station RANSAC R²')
ax2.set_title('Median RANSAC R² by Region\n(F1 + Physics Correction)', fontweight='bold')
ax2.set_ylim(0, max(med_r2.max() * 1.3, 0.1))

plt.suptitle('MODIS AOD 0.55 µm (F1, Physics-Corrected) vs PM2.5 — Per-Station Agreement',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 17 · Seasonal AOD–PM2.5 Correlation by Region

Monthly R² per region to see whether the AOD–PM2.5 relationship is stable year-round  
or collapses in certain seasons (e.g. wet season cloud contamination, biomass burning peaks).

In [ ]:
aqi_f1['month'] = pd.to_datetime(aqi_f1['timestamp']).dt.month

fig, ax = plt.subplots(figsize=(14, 5))

for region in REGIONS:
    color  = REGION_COLORS[region]
    subset = aqi_f1[aqi_f1['region'] == region]
    monthly_r2, months = [], []
    for m, grp in subset.groupby('month'):
        sub = grp[[AOD_COL_CORR, 'PM2.5']].dropna()
        if len(sub) < 10:
            continue
        try:
            fit = fit_ransac(sub[AOD_COL_CORR].values, sub['PM2.5'].values)
            monthly_r2.append(fit['r2'])
            months.append(m)
        except Exception:
            pass
    if months:
        ax.plot(months, monthly_r2, marker='o', color=color,
                lw=2.2, ms=8, label=region)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('RANSAC R²  (AOD_corr vs PM2.5)')
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', lw=1, linestyle='--', alpha=0.6, label='R²=0.5 guide')
ax.set_title(
    'Monthly RANSAC R² by Region\n'
    'MODIS AOD 0.55 µm (F1, Physics-Corrected) vs PM2.5',
    fontweight='bold', fontsize=12
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()